# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"{getattr(dataset.metadata, 'name', metadata_json.get('name', '<Unknown Dataset>'))}: {getattr(dataset.metadata, 'description', metadata_json.get('description', '<No description>'))}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

According to the Croissant schema, record sets are defined using their unique `@id`.

Let's enumerate available record sets and their fields using `mlcroissant`.

In [ ]:
# List all record sets and their fields by @id
record_sets = []
record_set_info = []

for rs in dataset.metadata.record_sets:
    record_sets.append(rs.id)
    print(f"Record set @id: {rs.id}, name: {getattr(rs, 'name', '')}")
    print("Fields:")
    for field in rs.fields:
        print(f"  - Field @id: {field.id}, name: {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', '')}")
        record_set_info.append({'record_set_id': rs.id, 'field_id': field.id, 'field_name': getattr(field, 'name', '')})
    print('-'*40)
if not record_sets:
    print("No record sets found in the Croissant metadata. Please check the schema structure.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract data from each record set (by `@id`) found above.

In [ ]:
# If there are record sets, extract data into DataFrames

dataframes = {}
if record_sets:
    for record_set_id in record_sets:
        print(f"Extracting records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}:", list(df.columns))
        display(df.head())
else:
    print("No record sets available. Extraction skipped.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll demonstrate with the first available record set and a numeric field (if available).

In [ ]:
# EDA example: filtering, normalization, grouping
import numpy as np

# Pick the first available record set for demonstration
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    # Find a numeric field (float or integer)
    numeric_field = None
    candidate_fields = []
    for rs in dataset.metadata.record_sets:
        if rs.id == record_set_id:
            for field in rs.fields:
                if getattr(field, 'data_type', '').lower() in ['float', 'integer', 'number']:
                    candidate_fields.append(field.id)
    if candidate_fields:
        numeric_field = candidate_fields[0]
        print(f"Using numeric field: {numeric_field}")
        if numeric_field in df.columns:
            # Choose an arbitrary threshold, e.g., the mean
            threshold = df[numeric_field].astype(float).mean()
            filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
            display(filtered_df.head())
            # Normalize
            fval = filtered_df[numeric_field].astype(float)
            filtered_df[f"{numeric_field}_normalized"] = (fval - fval.mean()) / (fval.std() if fval.std() else 1)
            print(f"Normalized values for {numeric_field}:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Find a possible group field (categorical)
            group_field = None
            for col in df.columns:
                if col != numeric_field and df[col].nunique() < 20:
                    group_field = col
                    break
            if group_field:
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by {group_field}:")
                display(grouped_df.head())
            else:
                print("No suitable group field found (categorical with <20 unique values).")
        else:
            print(f"Field '{numeric_field}' not found in DataFrame columns.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the chosen numeric field for the first record set, if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and 'numeric_field' in locals() and numeric_field:
    df = dataframes[record_set_id]
    if numeric_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].astype(float), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()
    else:
        print(f"Cannot plot: field '{numeric_field}' not found in DataFrame columns.")
else:
    print("No numeric field available for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the Mass Spectrometry dataset using the `mlcroissant` library, examined its structure, extracted records by their `@id`, and performed exploratory analysis, including normalization and visualizations. Further in-depth analysis can leverage the rich metadata and flexible extraction provided by Croissant via `mlcroissant`.